In [1]:
%matplotlib qt
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
from scipy import stats
import warnings

plt.rcParams['font.size'] = 14

# Given

In [2]:
# Path to files
dir_aper = os.path.expanduser('~/research/results/wash-u/aperiodicity')
dir_results = os.path.expanduser('~/research/results/wash-u/')
fname_aper = "aperiodicity_comparison.csv"
fpath_aper = os.path.join(dir_aper, fname_aper)

# Compare relative aperiodicity

In [3]:
# Read rel aper from file
df_aper = pd.read_csv(fpath_aper)
df_aper

,FilePathRaw,EyesClosedEpoch,RelativeAperiodicity_FFT,RelativeAperiodicity_HHT,Offset_FFT,Offset_HHT,Exponent_FFT,Exponent_HHT,Knee_FFT,Knee_HHT
0,C:\Users\chholakp2/data/wash-u/preprocessed/v2...,EC1,0.435353,0.417177,3.469985,-0.358249,1.632558,1.770016,0,0
1,C:\Users\chholakp2/data/wash-u/preprocessed/v2...,EC2,0.483814,0.452435,3.536177,-0.305455,1.654227,1.793012,0,0
2,C:\Users\chholakp2/data/wash-u/preprocessed/v2...,EC1,0.475105,0.397101,3.585260,-0.408711,1.630094,1.606541,0,0
3,C:\Users\chholakp2/data/wash-u/preprocessed/v2...,EC2,0.536839,0.476620,3.687541,-0.208967,1.799528,1.948214,0,0
4,C:\Users\chholakp2/data/wash-u/preprocessed/v2...,EC1,0.507075,0.511278,3.266028,-0.547182,1.578699,1.680962,0,0
...,...,...,...,...,...,...,...,...,...,...
407,C:\Users\chholakp2/data/wash-u/preprocessed/v2...,EC2,0.727888,0.854387,3.825558,0.083492,1.645251,1.793743,0,0
408,C:\Users\chholakp2/data/wash-u/preprocessed/v2...,EC1,0.664567,0.709901,3.467598,-0.335432,1.347158,1.426970,0,0
409,C:\Users\chholakp2/data/wash-u/preprocessed/v2...,EC2,0.638201,0.690476,3.483345,-0.305365,1.382614,1.492968,0,0
410,C:\Users\chholakp2/data/wash-u/preprocessed/v2...,EC1,0.524521,0.522775,3.559803,-0.325617,1.063369,1.087868,0,0


In [4]:
# Test whether rel aper values are normally distributed
apers_mode = {}
for mode in ['FFT', 'HHT']:
    apers_mode[mode] = df_aper['RelativeAperiodicity_' + mode]
    _, pval_shapirowilk = stats.shapiro(apers_mode[mode])
    if pval_shapirowilk > 0.05:
        warnings.warn('Shapiro-Wilk: Relative aperiodicity values for %s ARE normally distributed (p = %0.3f)' % (mode, pval_shapirowilk))
    else:
        print('Shapiro-Wilk: Relative aperiodicity values for %s ARE NOT normally distributed (p = %0.3f)' % (mode, pval_shapirowilk))

# Perform Wilcoxon Rank-Sum Test to check whether group medians of rel aperiodicities are different
res = stats.ranksums(apers_mode['FFT'], apers_mode['HHT'])
pval_wilcoxon = res.pvalue
if pval_wilcoxon > 0.05:
    print("Wilcoxon Rank-Sum: The group medians of Relative Aperiodicity dists of FFT and HHT ARE NOT different (p-val = %.3f)." % pval_wilcoxon)
else:
    print("Wilcoxon Rank-Sum: The group medians of Relative Aperiodicity dists of FFT and HHT ARE different (p-val = %.3f)." % pval_wilcoxon)

Shapiro-Wilk: Relative aperiodicity values for FFT ARE NOT normally distributed (p = 0.012)
Shapiro-Wilk: Relative aperiodicity values for HHT ARE NOT normally distributed (p = 0.000)
Wilcoxon Rank-Sum: The group medians of Relative Aperiodicity dists of FFT and HHT ARE NOT different (p-val = 0.662).


In [5]:
# Visualize
fname_fig = "AperiodicityHistogramsComparison.pdf"
fpath_fig = os.path.join(dir_aper, fname_fig)
plt.figure()
for mode in ['FFT', 'HHT']:
    plt.hist(apers_mode[mode], alpha=0.5, bins=np.linspace(0, 1.3, 14), label=mode)
plt.xlabel('Relative Aperiodicity')
plt.ylabel('Counts')
plt.legend()
plt.savefig(fpath_fig, bbox_inches='tight')
plt.savefig(fpath_fig[:-4] + '.png', bbox_inches='tight') # also save as png
plt.show()

#### Study HHT of subjects with relative aperiodicity > 1

In [31]:
# from fooof import FOOOF
import mne
from neuronol_signalprocessing import hilbert_spectra_from_raw

In [21]:
inds_unusual = np.where(df_aper['RelativeAperiodicity_HHT'] > 1)[0]
inds_unusual

array([ 73, 146, 149], dtype=int64)

In [9]:
df_aper['EyesClosedEpoch'][73]

'EC2'

In [50]:
# mean_specs = []
for i in inds_unusual:

    # Extract epoch identifiers
    fpath_raw = df_aper['FilePathRaw'][i]
    ec = df_aper['EyesClosedEpoch'][i]

    print(i)

    # Read preprocessed EEG data
    raw = mne.io.read_raw_fif(fpath_raw)

    # Find intervals for eyes closed resting state
    ec_end = [raw.annotations.onset[i] for i, ant_name in
                enumerate(raw.annotations.description) if ant_name in ('1', '3')]
    ec_intvl = {}
    ec_intvl['EC1'] = np.floor([ec_end[0] - 63, ec_end[0] - 3])
    ec_intvl['EC2'] = np.floor([ec_end[1] - 63, ec_end[1] - 3])
    tmin, tmax = ec_intvl[ec]

    # Crop raw
    cropped_raw = raw.copy().crop(tmin=tmin, tmax=tmax)

    # Show cropped raw
    cropped_raw.plot()

    # # Calculate power spectra
    # specs_mean_epochs, freqs = hilbert_spectra_from_raw(cropped_raw, compute_power_spec=True)
    # mean_spec = np.mean(specs_mean_epochs, axis=0)
    # mean_specs.append(mean_spec)


73
Opening raw data file C:\Users\chholakp2/data/wash-u/preprocessed/v2\7164_4_rest1_ec_ica_ssp_eeg.fif...
Isotrak not found
    Read a total of 2 projection items:
        ECG-eeg--0.200-0.400-PCA-01 (1 x 30) active
        ECG-eeg--0.200-0.400-PCA-02 (1 x 30) active
    Range : 0 ... 157499 =      0.000 ...   314.998 secs
Ready.
146
Opening raw data file C:\Users\chholakp2/data/wash-u/preprocessed/v2\7270_4_rest2_ec_ica_ssp_eeg.fif...
Isotrak not found
    Read a total of 2 projection items:
        ECG-eeg--0.200-0.400-PCA-01 (1 x 30) active
        ECG-eeg--0.200-0.400-PCA-02 (1 x 30) active
    Range : 0 ... 155839 =      0.000 ...   311.678 secs
Ready.
149
Opening raw data file C:\Users\chholakp2/data/wash-u/preprocessed/v2\7270_5_rest1_ec_ica_ssp_eeg.fif...
Isotrak not found
    Read a total of 2 projection items:
        ECG-eeg--0.200-0.400-PCA-01 (1 x 30) active
        ECG-eeg--0.200-0.400-PCA-02 (1 x 30) active
    Range : 0 ... 156699 =      0.000 ...   313.398 secs
Ready.

Channels marked as bad:
none
Channels marked as bad:
none
Channels marked as bad:
none


In [49]:
cropped_raw.plot()

Using qt as 2D backend.


Channels marked as bad:
none


In [48]:
for i_spec, i_raw in enumerate(inds_unusual):

    print(i_spec, i_raw)

    # Calculate aperiodic fit in linear spacing
    b = df_aper['Offset_HHT'][i_raw]; chi = df_aper['Exponent_HHT'][i_raw]; k = df_aper['Knee_HHT'][i_raw]
    lambda_f = (10 ** b) * 1 / (k + np.power(freqs, chi))

    # Visualize
    plt.figure()
    plt.plot(freqs, mean_specs[i_spec], label=r'$\psi (f)$')
    plt.plot(freqs, lambda_f, label=r'$\lambda (f)$')
    plt.xlim([0, 40])
    plt.legend()
    plt.show()

0 73
1 146
2 149


In [35]:
# # FOOOF Params
# fooof_params = {
#     'peak_width_limits': [2, 12],
#     'max_n_peaks': 6,
#     'min_peak_height': 0,
#     'peak_threshold': 2,
#     'aperiodic_mode': 'fixed'
# }
# freq_range = [50 / 60, 40]

# # Initialize FOOOF object
# fm = FOOOF(
#     peak_width_limits=fooof_params['peak_width_limits'],
#     max_n_peaks=fooof_params['max_n_peaks'],
#     min_peak_height=fooof_params['min_peak_height'],
#     peak_threshold=fooof_params['peak_threshold'],
#     aperiodic_mode=fooof_params['aperiodic_mode'],
# )

# # Fit the power spectrum model in the given `freq_range`
# fm.fit(freqs, mean_spec, freq_range)
# fm.report()

                                                                                                  
                                   FOOOF - POWER SPECTRUM MODEL                                   
                                                                                                  
                        The model was run on the frequency range 0 - 40 Hz                        
                                 Frequency Resolution is 0.02 Hz                                  
                                                                                                  
                            Aperiodic Parameters (offset, exponent):                              
                                          0.0059, 1.5638                                          
                                                                                                  
                                       2 peaks were found:                                        
          